사전 훈련된 임베딩 벡터를 가져오기

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 51.2 MB/s  0:00:00 eta 0:00:01


## 직접 훈련시킨 경우
먼저 사전 훈련된 임베딩을 사용하지 **않는** 경우부터 보자

모델의 긍,부정을 판단하는 감성 분류 모델

In [2]:
import numpy as np
from collections import Counter
import gensim

In [3]:
sentences = ['nice great best amazing', 'stop lies', 'pitiful nerd', 'excellent work',
             'supreme quality', 'bad', 'highly respectable']
y_train = [1, 0, 0, 1, 1, 0, 1] #긍정1, 부정0

In [4]:
tokenized_sentences = [sent.split() for sent in sentences]
print('단어 토큰화 된 결과 : ', tokenized_sentences)

단어 토큰화 된 결과 :  [['nice', 'great', 'best', 'amazing'], ['stop', 'lies'], ['pitiful', 'nerd'], ['excellent', 'work'], ['supreme', 'quality'], ['bad'], ['highly', 'respectable']]


In [5]:
word_list = []
for sent in tokenized_sentences:
    for word in sent:
        word_list.append(word)

word_counts = Counter(word_list)
print('총 단어 수 : ', len(word_counts))

총 단어 수 :  15


In [6]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print(vocab)

['nice', 'great', 'best', 'amazing', 'stop', 'lies', 'pitiful', 'nerd', 'excellent', 'work', 'supreme', 'quality', 'bad', 'highly', 'respectable']


In [7]:
word_to_index = {}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

for index, word in enumerate(vocab):
    word_to_index[word] = index + 2

vocab_size = len(word_to_index)
print('패딩 토큰, UNK토큰을 고려한 단어 집합의 크기 : ', vocab_size)
print(word_to_index)

패딩 토큰, UNK토큰을 고려한 단어 집합의 크기 :  17
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


In [8]:
# 정수 인코딩 진행
def texts_to_sequences(tokenized_X_data, word_to_index):
    encoded_X_data = []
    for sent in tokenized_X_data:
        index_sequences = []
        for word in sent:
            try:
                index_sequences.append(word_to_index[word])
            except KeyError:
                index_sequences.append(word_to_index['<UNK>'])
        encoded_X_data.append(index_sequences)
    return encoded_X_data

X_encoded = texts_to_sequences(tokenized_sentences, word_to_index)
print(X_encoded)

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]


In [9]:
max_len = max(len(l) for l in X_encoded)
print('최대 길이 : ', max_len)

최대 길이 :  4


In [10]:
def pad_seqences(sentences, max_len):
    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index, :len(sentence)] = np.array(sentence)[:max_len]
    return features

X_train = pad_seqences(X_encoded, max_len)
y_train = np.array(y_train)
print('패딩 결과 :')
print(X_train)

패딩 결과 :
[[ 2  3  4  5]
 [ 6  7  0  0]
 [ 8  9  0  0]
 [10 11  0  0]
 [12 13  0  0]
 [14  0  0  0]
 [15 16  0  0]]


In [11]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

In [12]:
class SimpleModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SimpleModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(embedding_dim * max_len, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        flattened = self.flatten(embedded)

        output = self.fc(flattened)
        return self.sigmoid(output)
    

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

embedding_dim = 100
model = SimpleModel(vocab_size, embedding_dim).to(device)

In [14]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters())

In [15]:
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.float))
train_dataloader = DataLoader(train_dataset, batch_size=2)

In [16]:
print(len(train_dataloader))

4


In [17]:
for epoch in range(10):
    for input, targets in train_dataloader:
        inputs, targets = input.to(device), targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs).view(-1)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/10], Loss: {loss.item():.4f}')


Epoch [1/10], Loss: 0.8960
Epoch [2/10], Loss: 0.6875
Epoch [3/10], Loss: 0.5072
Epoch [4/10], Loss: 0.3800
Epoch [5/10], Loss: 0.2977
Epoch [6/10], Loss: 0.2463
Epoch [7/10], Loss: 0.2142
Epoch [8/10], Loss: 0.1932
Epoch [9/10], Loss: 0.1780
Epoch [10/10], Loss: 0.1654
